# 🛰️ NGEO Satellite Downloader (NGEO / IDEFLOR-Bio)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NGEO-IDEFLOR-Bio/ngeo_satellite_downloader/blob/main/ngeo_satellite_downloader.ipynb)

Ferramenta operacional automatizada do **Núcleo de Geotecnologias (NGEO) do Instituto de Desenvolvimento Florestal e da Biodiversidade do Estado do Pará (IDEFLOR-Bio)** para download de imagens de satélite sobre áreas de interesse definidas por Shapefile.

---

### 🛰️ Sensores Disponíveis e Características

| Sensor | Resolução | Particularidades e Aplicações |
|---|---:|---|
| **Sentinel-2 MSI** | 10 m | Composição semestral ou mensal com máscara de nuvens e sombras via SCL (Scene Classification). |
| **Landsat 5 / 7 / 8 / 9** | 30 m | Séries históricas calibradas (Surface Reflectance), máscara QA_PIXEL e seleção por semestre. |
| **CBERS-4A (WPM/MUX)** | 2 m | Alta resolução espacial nacional integrada à API do INPE com composição RGBN e recorte exato. |
| **SPOT 2008** | 5 m | Mosaico histórico de alta resolução de referência do Código Florestal Brasileiro (MMA/SFB). |

---

### 🚀 Fluxo de Trabalho (Interface Simplificada)
1. **Ambiente**: Execute o passo 1 para preparar os pacotes e carregar o motor cartográfico.
2. **Autenticação**: Confirme seu projeto no Earth Engine e autorize a montagem do Google Drive.
3. **Shapefile**: Faça o upload do arquivo `.zip` da sua área de estudo para validação automática.
4. **Parâmetros**: Selecione o sensor, período (anos/meses/semestres), método e buffer de expansão.
5. **Download**: Inicie a exportação em alta resolução GeoTIFF direto para o seu Google Drive.

In [ ]:
#@title 1. Instalação de Dependências e Setup do Ambiente { display-mode: "form" }
#@markdown *Clique no botão 'Play' para preparar o ambiente e carregar o motor cartográfico do NGEO.*

!pip install -q geemap cbers4asat rasterio geopandas pycrs geomet geojson scikit-image requests pillow

import os
import sys
from pathlib import Path

REPO_NAME = "ngeo_satellite_downloader"
if not Path(REPO_NAME).exists() and not Path("src").exists():
    print(f"Clonando repositório oficial {REPO_NAME}...")
    !git clone https://github.com/NGEO-IDEFLOR-Bio/ngeo_satellite_downloader.git

if Path(REPO_NAME).exists():
    os.chdir(REPO_NAME)

sys.path.append(os.getcwd())

import src
print(f"[OK] Ambiente configurado com sucesso! Versão do motor: {src.__version__}")


In [ ]:
#@title 2. Autenticação no Google Earth Engine (GEE) e Google Drive { display-mode: "form" }
GCP_PROJECT = "sentinel-landsat-download" #@param {type:"string"}
MONTAR_GOOGLE_DRIVE = True #@param {type:"boolean"}
#@markdown > ⚠️ **Atenção colegas:** Altere o campo `GCP_PROJECT` acima para o ID do seu próprio projeto Google Cloud habilitado no Earth Engine caso utilize uma conta institucional pessoal.

import os
import ee
from google.colab import drive

# 1. Inicializar Google Earth Engine
try:
    ee.Initialize(project=GCP_PROJECT)
    print(f"[OK] Google Earth Engine inicializado com sucesso no projeto: {GCP_PROJECT}")
except Exception as e:
    print("! Solicitando autenticação interativa no Google Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project=GCP_PROJECT)
    print(f"[OK] Earth Engine autenticado e inicializado no projeto: {GCP_PROJECT}!")

# 2. Montar Google Drive
if MONTAR_GOOGLE_DRIVE:
    try:
        drive.mount('/content/drive')
        print("[OK] Google Drive montado com sucesso em /content/drive")
    except Exception as e:
        print(f"[AVISO] Não foi possível montar o Google Drive automaticamente: {e}")


In [ ]:
#@title 3. Upload da Área de Interesse (Shapefile .zip) { display-mode: "form" }
#@markdown *Clique no 'Play' para selecionar e enviar o arquivo .ZIP com seu shapefile (.shp, .shx, .dbf, .prj).*
#@markdown *A geometria será projetada automaticamente para SIRGAS 2000 / WGS-84 (EPSG:4326).*

import os
import io
import shutil
from google.colab import files
from src.vector_ops import extract_shapefile_from_zip, load_shapefile_geometry, calculate_aoi_stats
from src.ui_helpers import render_aoi_card

print(">> Selecione o arquivo .ZIP com o shapefile...")
uploaded = files.upload()

SHAPEFILE_GEOMETRY = None
GDF_AOI = None
AOI_STATS = None

if uploaded:
    zip_name = list(uploaded.keys())[0]
    zip_bytes = uploaded[zip_name]
    shp_path = extract_shapefile_from_zip(zip_bytes, extract_dir="shapefile")
    print(f"[OK] Shapefile extraído: {shp_path}")
    
    SHAPEFILE_GEOMETRY, GDF_AOI = load_shapefile_geometry(shp_path)
    AOI_STATS = calculate_aoi_stats(SHAPEFILE_GEOMETRY, GDF_AOI)
    render_aoi_card(AOI_STATS)
else:
    shp_fallback = [os.path.join("shapefile", f) for f in os.listdir("shapefile") if f.endswith(".shp")] if os.path.exists("shapefile") else []
    if shp_fallback:
        print(f"[INFO] Utilizando shapefile pré-existente: {shp_fallback[0]}")
        SHAPEFILE_GEOMETRY, GDF_AOI = load_shapefile_geometry(shp_fallback[0])
        AOI_STATS = calculate_aoi_stats(SHAPEFILE_GEOMETRY, GDF_AOI)
        render_aoi_card(AOI_STATS)
    else:
        print("[AVISO] Nenhum arquivo foi enviado. Por favor, execute a célula e selecione um arquivo .zip.")


In [ ]:
#@title 3.1 (Opcional) Visualizar Área no Mapa Interativo { display-mode: "form" }
#@markdown *Gera um mapa interativo com o polígono carregado e basemap de satélite.*

import geemap
from IPython.display import display

if "SHAPEFILE_GEOMETRY" in globals() and SHAPEFILE_GEOMETRY is not None:
    centro = AOI_STATS.get("center", [-3.5, -52.0]) if AOI_STATS else [-3.5, -52.0]
    Map = geemap.Map(center=centro, zoom=10)
    Map.add_basemap("SATELLITE")
    Map.addLayer(SHAPEFILE_GEOMETRY, {"color": "red", "fillColor": "00000000", "width": 2}, "Área de Interesse")
    display(Map)
else:
    print("[AVISO] Carregue primeiro o shapefile no passo 3.")


In [ ]:
#@title 4. Parâmetros de Download e Configuração do Sensor { display-mode: "form" }
SATELITE = "Sentinel-2 (10m)" #@param ["Sentinel-2 (10m)", "Landsat 5/7/8/9 (30m)", "CBERS-4A (MUX/WPM 2m)", "SPOT 2008 - Mosaico CF (5m)"]
ANOS = "2024" #@param {type:"string"}
#@markdown *Informe um ano único (`2024`), um intervalo (`2020-2024`) ou lista (`2021, 2023, 2024`).*

MODO = "Composicao por Semestre" #@param ["Composicao por Semestre", "Analise Anual (Meses Selecionados)"]
SEMESTRE = "Ambos" #@param ["1º Semestre (Jan-Jun)", "2º Semestre (Jul-Dez)", "Ambos"]
MESES_ANALISE = "6, 7, 8" #@param {type:"string"}
#@markdown *Para o modo Análise Anual, informe os números dos meses separados por vírgula (ex: `6, 7, 8` para Jun, Jul e Ago).*

METODO = "median" #@param [("Mediana Inteligente (Recomendado - Mais limpa)", "median"), ("Melhor Imagem (Menor % de nuvens)", "best")]
BUFFER = 2 #@param {type:"slider", min:1.0, max:5.0, step:0.5}
#@markdown *Buffer de ampliação da área (1.0 = extensão exata; 2.0 = margem padrão de segurança no entorno).*

PASTA_DRIVE = "Satellite_Downloads" #@param {type:"string"}
#@markdown *Nome da pasta no Google Drive para armazenamento dos GeoTIFFs.*

# Credenciais INPE (obrigatório apenas se selecionar CBERS-4A)
INPE_EMAIL = "" #@param {type:"string"}
INPE_SENHA = "" #@param {type:"raw"}

from src.vector_ops import parse_years

anos_lista = parse_years(ANOS)
meses_lista = [int(m.strip()) for m in MESES_ANALISE.split(',') if m.strip().isdigit()]

print("Configurações validadas:")
print(f" • Sensor selecionado: {SATELITE}")
print(f" • Anos para processamento: {anos_lista}")
print(f" • Modo de análise: {MODO}")
if MODO == "Composicao por Semestre":
    print(f" • Semestre: {SEMESTRE}")
else:
    print(f" • Meses selecionados: {meses_lista}")
print(f" • Método de composição: {METODO}")
print(f" • Buffer de expansão: {BUFFER}x")
print(f" • Pasta de destino no Drive: {PASTA_DRIVE}")


In [ ]:
#@title 5. Executar Download e Exportação para o Google Drive { display-mode: "form" }
#@markdown *Clique no 'Play' para iniciar as consultas orbitais, composição com máscara de nuvens e exportação.*

import sys
from src.exporter import run_pipeline

if "SHAPEFILE_GEOMETRY" not in globals() or SHAPEFILE_GEOMETRY is None:
    raise ValueError("Por favor, execute primeiro o passo 3 (Upload do Shapefile) antes de iniciar o download!")

resultado = run_pipeline(
    region=SHAPEFILE_GEOMETRY,
    satellite=SATELITE,
    years=anos_lista,
    mode=MODO,
    semester=SEMESTRE,
    months=meses_lista,
    method=METODO,
    buffer_factor=float(BUFFER),
    drive_folder=PASTA_DRIVE,
    inpe_email=INPE_EMAIL.strip() if INPE_EMAIL else None
)


In [ ]:
#@title 6. Resumo dos Arquivos Gerados e Acesso aos Downloads { display-mode: "form" }

from src.ui_helpers import render_results_cards

if "resultado" in globals() and resultado:
    render_results_cards(resultado)
else:
    print("[AVISO] Nenhuma exportação foi executada ainda. Execute a etapa 5 primeiro.")


---

### 🛡️ Boas Práticas e Controle de Qualidade (NGEO / IDEFLOR-Bio)
- **Verificação Espacial**: Abra os arquivos GeoTIFF baixados no QGIS e confira a projeção cartográfica (padrão SIRGAS 2000 ou WGS-84).
- **Inspeção de Nuvens**: Para áreas equatoriais (Amazônia), verifique se há artefatos residuais em períodos chuvosos (Jan a Mai). Se necessário, utilize o modo **Análise Anual** filtrando os meses de menor nebulosidade (Jul a Out).
- **Divisão em Tiles**: Para extensões acima de 5 milhões de pixels, o mosaico gerado é dividido em quadrículas para garantir resolução nativa sem perda de amostragem.

Desenvolvimento e coordenação: [**Samuel da Costa dos Santos**](https://samuelsantos.site/)  
**Núcleo de Geotecnologias (NGEO) — Instituto de Desenvolvimento Florestal e da Biodiversidade do Estado do Pará (IDEFLOR-Bio)**
